# Test `generate_building_boundary` and prepare a Grasshopper handoff

This notebook does three things:
1. Runs the local Python footprint generator.
2. Inspects the returned boundary coordinates and metrics.
3. Builds a JSON payload that a Grasshopper-side import tool can consume through Swiftlet MCP.

## Expected handoff architecture

Keep the initial footprint generator in Python. Then add a separate Grasshopper import tool that accepts polygon coordinates and creates a Rhino polyline or curve.

Recommended data contract:
```json
{
  "geometry_id": "generate_building_boundary_xxx",
  "building_footprint": {
    "type": "Polygon",
    "coordinates": [[x, y, z], [x, y, z], ...]
  },
  "metadata": {
    "shape_type": "I",
    "boundary_area_sqm": 900.0
  }
}
```

In [5]:
# import sys
# import subprocess

# print(sys.executable)
# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "-U",
#     "langchain-core",
#     "langchain-openai",
#     "openai",
# ])

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / 'team_04',
    workspace_root.parent / 'team_04',
)
TEAM_ROOT = next((path for path in candidate_roots if (path / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run this notebook from the workspace root, the team_04 folder, or the team_04/notebooks folder.')

team_root_str = str(TEAM_ROOT)
if team_root_str not in sys.path:
    sys.path.insert(0, team_root_str)

OUTPUT_JSON = TEAM_ROOT / 'notebooks' / 'generated_building_boundary_payload.json'
SWIFTLET_MCP_URL = 'http://localhost:3001/mcp/'
GH_IMPORT_TOOL_NAME = 'import_building_boundary'

In [ ]:
from __future__ import annotations

import importlib
import os

from langchain_openai import ChatOpenAI

state_module = importlib.reload(importlib.import_module("agent.state"))
decision_engine_module = importlib.reload(importlib.import_module("agent.decision_engine"))
graph_module = importlib.reload(importlib.import_module("agent.graph"))
generate_boundary_module = importlib.reload(importlib.import_module("agent.tools.generate_building_boundary"))
tools_module = importlib.reload(importlib.import_module("agent.tools"))
mcp_client_module = importlib.reload(importlib.import_module("agent.mcp_client"))
tool_catalog_module = importlib.reload(importlib.import_module("agent.tool_catalog"))

OpenAIDecisionEngine = decision_engine_module.OpenAIDecisionEngine
RuleBasedPlanner = decision_engine_module.RuleBasedPlanner
run_agent = graph_module.run_agent
LocalToolClient = mcp_client_module.LocalToolClient
CompositeToolClient = mcp_client_module.CompositeToolClient
build_default_local_tool_client = mcp_client_module.build_default_local_tool_client
ToolCatalog = tool_catalog_module.ToolCatalog

USER_BOUNDARY_BRIEF = (
    "Generate an L-shaped building boundary for a site area of 5000 square meters. "
    "If the exact building size is not specified, use the tool's default planning assumption."
)
SITE_AREA_SQM = 5000.0
SITE_WIDTH_M = 100.0
SITE_DEPTH_M = SITE_AREA_SQM / SITE_WIDTH_M
SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [SITE_WIDTH_M, 0.0, 0.0],
    [SITE_WIDTH_M, SITE_DEPTH_M, 0.0],
    [0.0, SITE_DEPTH_M, 0.0],
    [0.0, 0.0, 0.0],
]

def site_boundary_reader(layout_json: str = "") -> dict:
    del layout_json
    return {
        "success": True,
        "data": {
            "site_boundary": SITE_BOUNDARY,
            "site_area_sqm": SITE_AREA_SQM,
        },
    }

def context_reader(layout_json: str = "") -> dict:
    del layout_json
    return {
        "success": True,
        "data": {
            "summary": "Simple notebook site context.",
            "site_area_sqm": SITE_AREA_SQM,
        },
    }

def legal_constraints_reader(layout_json: str = "") -> dict:
    del layout_json
    return {
        "success": True,
        "data": {
            "setback_m": 5.0,
        },
    }

site_tool_client = LocalToolClient(
    {
        "site_boundary_reader": (
            {"name": "site_boundary_reader", "description": "Read the site boundary for the current notebook site."},
            site_boundary_reader,
        ),
        "context_reader": (
            {"name": "context_reader", "description": "Read high-level site context for the current notebook site."},
            context_reader,
        ),
        "legal_constraints_reader": (
            {"name": "legal_constraints_reader", "description": "Read legal constraints for the current notebook site."},
            legal_constraints_reader,
        ),
    }
)

tool_client = CompositeToolClient([build_default_local_tool_client(), site_tool_client])
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())
decision_engine = OpenAIDecisionEngine(
    llm=ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=0,
    )
)

agent_state = run_agent(
    user_prompt=USER_BOUNDARY_BRIEF,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout={
        "site_boundary": SITE_BOUNDARY,
        "target_building_count": 1,
        "workflow_mode": "boundary_only",
    },
    max_optimization_cycles=0,
    planner=RuleBasedPlanner(),
)

result = next(
    record["output"]
    for record in reversed(agent_state.get("tool_history", []))
    if isinstance(record, dict) and record.get("tool") == "generate_building_boundary"
    and isinstance(record.get("output"), dict)
    and isinstance(record["output"].get("data", {}).get("boundary"), list)
    )

decision_trace = [
    message
    for message in agent_state.get("messages", [])
    if message.startswith(("Planner updated", "Supervisor decision", "Tool ", "Final report"))
]

print("Agent decision trace:")
for index, message in enumerate(decision_trace, start=1):
    print(f"{index}. {message}")

print("\nFinal report:")
print(agent_state.get("final_response", ""))

{
    "workflow_mode": agent_state.get("workflow_mode"),
    "final_response": agent_state.get("final_response"),
    "decision_trace": decision_trace,
    "selected_geometry_id": result["data"]["geometry_id"],
    "selected_parameters": result["data"]["parameters"],
}

Agent decision trace:
1. Planner updated the task sequence: Initial planning required.
2. Supervisor decision: read_site | Load site boundary, context, and legal constraints.
3. Tool site_boundary_reader_04 executed.
4. Tool legal_constraints_reader_04 executed.
5. Tool context_reader_04 executed.
6. Planner updated the task sequence: read_site step finished.
7. Supervisor decision: generate_shape | Generate a candidate L-shaped (typology default) building footprint boundary for building 1 using the required area input; other parameters use tool defaults.
8. Tool generate_building_boundary executed.
9. Planner updated the task sequence: Generated a new geometry candidate.
10. Supervisor decision: report | Write the design report for the current best state.
11. Final report generated.

Final report:
### Chosen geometry (Building 1)
- **Geometry type:** L-shaped building boundary  
- **Geometry ID:** `generate_building_boundary_7af6dad30cbf`
- **Footprint boundary (XY, Z=0):**
  1. (-7.5

{'workflow_mode': 'boundary_only',
 'final_response': '### Chosen geometry (Building 1)\n- **Geometry type:** L-shaped building boundary  \n- **Geometry ID:** `generate_building_boundary_7af6dad30cbf`\n- **Footprint boundary (XY, Z=0):**\n  1. (-7.5, -7.5, 0)\n  2. (69.5, -7.5, 0)\n  3. (69.5, 7.5, 0)\n  4. (7.5, 7.5, 0)\n  5. (7.5, 54.666667, 0)\n  6. (-7.5, 54.666667, 0)\n  7. (-7.5, -7.5, 0)\n- **Planned footprint area (tool output):** **1862.5 sqm**  \n- **Bounding box:** min (-7.5, -7.5) to max (69.5, 54.666667)  \n\n### Remaining constraint state\n- **Requested position / placement checks:** *not performed* (skipped)\n- **Constraints validation (e.g., setbacks/inside lot):** *not performed* (skipped)\n- **Violations reported:** **none** (because constraint evaluation was not run)\n\n### Evaluation results\n- **Design/performance evaluation:** *not performed* (skipped)\n- **Evaluation metrics:** none recorded\n\n### Next recommendation\n1. **Run constraint validation** for Buildin

## Simple end-to-end agent run

This cell now uses the real Team 04 LangGraph agent in `boundary_only` mode.

The agent still calls OpenAI through `OpenAIDecisionEngine`, but the workflow is limited to:
1. read site
2. generate building boundary
3. report

That keeps the notebook small while still using the actual agent graph.

In [25]:
if "agent_state" not in globals() or "result" not in globals():
    raise RuntimeError("Run Cell 5 first to execute the end-to-end LangGraph agent.")

result

{'success': True,
 'data': {'geometry_id': 'generate_building_boundary_7af6dad30cbf',
  'shape_type': 'L',
  'boundary': [[-7.5, -7.5, 0.0],
   [69.5, -7.5, 0.0],
   [69.5, 7.5, 0.0],
   [7.5, 7.5, 0.0],
   [7.5, 54.666667, 0.0],
   [-7.5, 54.666667, 0.0],
   [-7.5, -7.5, 0.0]],
  'boundary_area_sqm': 1862.5,
  'perimeter_m': 278.333333,
  'centroid': [19.224161, 11.807494, 0.0],
  'bounding_box': {'min': [-7.5, -7.5, 0.0], 'max': [69.5, 54.666667, 0.0]},
  'parameters': {'area': 1750.0,
   'building_type': 'L',
   'building_depth': 15.0,
   'shape_ratio': 0.66,
   'location_xy': [0.0, 0.0],
   'is_mirrored': False,
   'max_rotation_angle': 180,
   'max_rotation_step': 4,
   'rotation_step': 0,
   'applied_rotation_angle': 0.0}},
 'metadata': {'tool_name': 'generate_building_boundary', 'source': 'python'}}

In [3]:
boundary = result['data']['boundary']
boundary[:5], len(boundary), result['data']['boundary_area_sqm'], result['data']['perimeter_m']

([[30.0, 7.272078, 0.0],
  [59.22708, 36.499158, 0.0],
  [46.499158, 49.22708, 0.0],
  [30.0, 32.727922, 0.0],
  [5.722667, 57.005255, 0.0]],
 7,
 1362.0,
 187.333333)

In [4]:
gh_payload = {
    'geometry_id': result['data']['geometry_id'],
    'building_footprint': {
        'type': 'Polygon',
        'coordinates': result['data']['boundary'],
    },
    'metadata': {
        'shape_type': result['data']['shape_type'],
        'boundary_area_sqm': result['data']['boundary_area_sqm'],
        'perimeter_m': result['data']['perimeter_m'],
        'centroid': result['data']['centroid'],
        'bounding_box': result['data']['bounding_box'],
        'generator_parameters': result['data']['parameters'],
        'source_tool': result['metadata']['tool_name'],
    },
}

OUTPUT_JSON.write_text(json.dumps(gh_payload, indent=2), encoding='utf-8')
gh_payload

{'geometry_id': 'generate_building_boundary_5c59d5a69051',
 'building_footprint': {'type': 'Polygon',
  'coordinates': [[30.0, 7.272078, 0.0],
   [59.22708, 36.499158, 0.0],
   [46.499158, 49.22708, 0.0],
   [30.0, 32.727922, 0.0],
   [5.722667, 57.005255, 0.0],
   [-7.005255, 44.277333, 0.0],
   [30.0, 7.272078, 0.0]]},
 'metadata': {'shape_type': 'L',
  'boundary_area_sqm': 1362.0,
  'perimeter_m': 187.333333,
  'centroid': [26.110913, 32.901843, 0.0],
  'bounding_box': {'min': [-7.005255, 7.272078, 0.0],
   'max': [59.22708, 57.005255, 0.0]},
  'generator_parameters': {'area': 1200.0,
   'building_type': 'L',
   'building_depth': 18.0,
   'shape_ratio': 0.62,
   'location_xy': [30.0, 20.0],
   'is_mirrored': False,
   'max_rotation_angle': 180.0,
   'max_rotation_step': 4,
   'rotation_step': 1,
   'applied_rotation_angle': 45.0},
  'source_tool': 'generate_building_boundary'}}

## Option A: live MCP import to Grasshopper

This notebook now treats the Grasshopper MCP import tool as the primary handoff path.

The active flow is:
1. Generate the footprint in Python.
2. Build the MCP request body for `import_building_boundary`.
3. Send the request to the Swiftlet MCP endpoint.
4. Read back the Rhino or Grasshopper result payload.

Expected behavior from `import_building_boundary`:
- accept the polygon coordinates from `gh_payload`
- create a closed Rhino polyline or Nurbs curve
- optionally bake it to a named layer
- return Rhino GUIDs and any derived metrics

Fallback note: file-based handoff is still possible, but this notebook is now focused on the live MCP path.

In [5]:
def build_mcp_tool_call_payload(tool_name: str, arguments: dict) -> dict:
    return {
        'jsonrpc': '2.0',
        'id': 1,
        'method': 'tools/call',
        'params': {
            'name': tool_name,
            'arguments': arguments,
        },
    }

mcp_arguments = {
    'geometry_id': gh_payload['geometry_id'],
    'boundary': gh_payload['building_footprint']['coordinates'],
    'shape_type': gh_payload['metadata']['shape_type'],
    'layer_name': 'TerraPilot_Output::BuildingFootprint',
    'closed': True,
}

mcp_request_body = build_mcp_tool_call_payload(GH_IMPORT_TOOL_NAME, mcp_arguments)
mcp_request_body

{'jsonrpc': '2.0',
 'id': 1,
 'method': 'tools/call',
 'params': {'name': 'import_building_boundary_04',
  'arguments': {'geometry_id': 'generate_building_boundary_5c59d5a69051',
   'boundary': [[30.0, 7.272078, 0.0],
    [59.22708, 36.499158, 0.0],
    [46.499158, 49.22708, 0.0],
    [30.0, 32.727922, 0.0],
    [5.722667, 57.005255, 0.0],
    [-7.005255, 44.277333, 0.0],
    [30.0, 7.272078, 0.0]],
   'shape_type': 'L',
   'layer_name': 'TerraPilot_Output::BuildingFootprint',
   'closed': True}}}

In [8]:
# Run this cell only after Rhino + Swiftlet are open and the Grasshopper import tool exists.
# This version prints the import tool response explicitly.

import socket
import urllib.error
import urllib.request

request = urllib.request.Request(
    SWIFTLET_MCP_URL,
    data=json.dumps(mcp_request_body).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)

try:
    with urllib.request.urlopen(request, timeout=60) as response:
        raw_response = response.read().decode('utf-8')
        bridge_result = json.loads(raw_response)
    print('Grasshopper import tool response:')
    print(json.dumps(bridge_result, indent=2))
    bridge_result
except urllib.error.URLError as exc:
    print('Swiftlet MCP request failed:', exc)
    print('If Rhino is open, the likely missing piece is the Grasshopper import tool implementation or endpoint availability.')
except socket.timeout:
    print('Swiftlet MCP request timed out after 60 seconds.')
    print('If the tool is running, inspect the Swiftlet bridge or Grasshopper tool execution path.')

Grasshopper import tool response:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "{\r\n  \"area\": 1361.9999906395724\r\n}"
      }
    ]
  }
}


## Grasshopper-side import tool sketch

Your Grasshopper bridge tool should accept `boundary` as a list of `[x, y, z]` points, convert them into Rhino points, create a closed polyline, and optionally bake it.

Minimum GH tool inputs:
- `geometry_id`: string
- `boundary`: array of `[x, y, z]` points
- `shape_type`: string
- `layer_name`: string
- `closed`: boolean

Minimum GH tool outputs:
- `geometry_id`: string
- `footprint_guid`: Rhino curve GUID
- `point_count`: integer
- `is_closed`: boolean
- `layer_name`: string

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import socket
import urllib.error
import urllib.request

from langchain_openai import ChatOpenAI

# User-editable prompt for quick testing.
USER_WORKFLOW_PROMPT = (
    "Generate an L-shaped building boundary with a building area of 3000 square meters. "
    "Rotate the building by 45 degrees. Use tool defaults for any unspecified shape parameters."
)

# Fixed placeholder site context for this boundary-only test.
SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [200.0, 0.0, 0.0],
    [200.0, 200.0, 0.0],
    [0.0, 200.0, 0.0],
    [0.0, 0.0, 0.0],
]

state_module = importlib.reload(importlib.import_module("agent.state"))
decision_engine_module = importlib.reload(importlib.import_module("agent.decision_engine"))
graph_module = importlib.reload(importlib.import_module("agent.graph"))
generate_boundary_module = importlib.reload(importlib.import_module("agent.tools.generate_building_boundary"))
tools_module = importlib.reload(importlib.import_module("agent.tools"))
mcp_client_module = importlib.reload(importlib.import_module("agent.mcp_client"))
tool_catalog_module = importlib.reload(importlib.import_module("agent.tool_catalog"))

OpenAIDecisionEngine = decision_engine_module.OpenAIDecisionEngine
RuleBasedPlanner = decision_engine_module.RuleBasedPlanner
run_agent = graph_module.run_agent
LocalToolClient = mcp_client_module.LocalToolClient
CompositeToolClient = mcp_client_module.CompositeToolClient
build_default_local_tool_client = mcp_client_module.build_default_local_tool_client
ToolCatalog = tool_catalog_module.ToolCatalog

def site_boundary_reader(layout_json: str = "") -> dict:
    del layout_json
    return {
        "success": True,
        "data": {
            "site_boundary": SITE_BOUNDARY,
            "site_area_sqm": 40000.0,
        },
    }

def context_reader(layout_json: str = "") -> dict:
    del layout_json
    return {
        "success": True,
        "data": {
            "summary": "Simple notebook site context.",
            "site_area_sqm": 40000.0,
        },
    }

def legal_constraints_reader(layout_json: str = "") -> dict:
    del layout_json
    return {
        "success": True,
        "data": {
            "setback_m": 5.0,
        },
    }

site_tool_client = LocalToolClient(
    {
        "site_boundary_reader": (
            {"name": "site_boundary_reader", "description": "Read the site boundary for the current notebook site."},
            site_boundary_reader,
        ),
        "context_reader": (
            {"name": "context_reader", "description": "Read high-level site context for the current notebook site."},
            context_reader,
        ),
        "legal_constraints_reader": (
            {"name": "legal_constraints_reader", "description": "Read legal constraints for the current notebook site."},
            legal_constraints_reader,
        ),
    }
)

tool_client = CompositeToolClient([build_default_local_tool_client(), site_tool_client])
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())
decision_engine = OpenAIDecisionEngine(
    llm=ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=0,
    )
)

agent_state = run_agent(
    user_prompt=USER_WORKFLOW_PROMPT,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout={
        "site_boundary": SITE_BOUNDARY,
        "target_building_count": 1,
        "workflow_mode": "boundary_only",
    },
    max_optimization_cycles=0,
    planner=RuleBasedPlanner(),
)

result = next(
    record["output"]
    for record in reversed(agent_state.get("tool_history", []))
    if isinstance(record, dict) and record.get("tool") == "generate_building_boundary"
    and isinstance(record.get("output"), dict)
    and isinstance(record["output"].get("data", {}).get("boundary"), list)
)

decision_trace = [
    message
    for message in agent_state.get("messages", [])
    if message.startswith(("Planner updated", "Supervisor decision", "Tool ", "Final report"))
]

gh_payload = {
    "geometry_id": result["data"]["geometry_id"],
    "building_footprint": {
        "type": "Polygon",
        "coordinates": result["data"]["boundary"],
    },
    "metadata": {
        "shape_type": result["data"]["shape_type"],
        "boundary_area_sqm": result["data"]["boundary_area_sqm"],
        "perimeter_m": result["data"]["perimeter_m"],
        "centroid": result["data"]["centroid"],
        "bounding_box": result["data"]["bounding_box"],
        "generator_parameters": result["data"]["parameters"],
        "source_tool": result["metadata"]["tool_name"],
    },
}

OUTPUT_JSON.write_text(json.dumps(gh_payload, indent=2), encoding="utf-8")

mcp_request_body = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": GH_IMPORT_TOOL_NAME,
        "arguments": {
            "geometry_id": gh_payload["geometry_id"],
            "boundary": gh_payload["building_footprint"]["coordinates"],
            "shape_type": gh_payload["metadata"]["shape_type"],
            "layer_name": "TerraPilot_Output::BuildingFootprint",
            "closed": True,
        },
    },
}

request = urllib.request.Request(
    SWIFTLET_MCP_URL,
    data=json.dumps(mcp_request_body).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

bridge_result = None
bridge_error = None

try:
    with urllib.request.urlopen(request, timeout=60) as response:
        raw_response = response.read().decode("utf-8")
        bridge_result = json.loads(raw_response)
except urllib.error.URLError as exc:
    bridge_error = f"Swiftlet MCP request failed: {exc}"
except socket.timeout:
    bridge_error = "Swiftlet MCP request timed out after 60 seconds."

print("User prompt:")
print(USER_WORKFLOW_PROMPT)

print("\nAgent decision trace:")
for index, message in enumerate(decision_trace, start=1):
    print(f"{index}. {message}")

print("\nAgent final report:")
print(agent_state.get("final_response", ""))

print("\nGenerated payload:")
print(json.dumps(gh_payload, indent=2))

print("\nMCP request body:")
print(json.dumps(mcp_request_body, indent=2))

if bridge_result is not None:
    print("\nGrasshopper import tool response:")
    print(json.dumps(bridge_result, indent=2))
else:
    print("\nGrasshopper import error:")
    print(bridge_error)

{
    "user_prompt": USER_WORKFLOW_PROMPT,
    "workflow_mode": agent_state.get("workflow_mode"),
    "decision_trace": decision_trace,
    "final_response": agent_state.get("final_response"),
    "gh_payload": gh_payload,
    "mcp_request_body": mcp_request_body,
    "bridge_result": bridge_result,
    "bridge_error": bridge_error,
}

User prompt:
Generate an L-shaped building boundary with a building area of 3000 square meters. Rotate the building by 45 degrees. Use tool defaults for any unspecified shape parameters.

Agent decision trace:
1. Planner updated the task sequence: Initial planning required.
2. Supervisor decision: read_site | Load site boundary, context, and legal constraints.
3. Tool site_boundary_reader_04 executed.
4. Tool legal_constraints_reader_04 executed.
5. Tool context_reader_04 executed.
6. Planner updated the task sequence: read_site step finished.
7. Supervisor decision: generate_shape | Generating building 1 footprint boundary with the requested area (3000 sqm) and a single 45° rotation using tool defaults for other shape parameters.
8. Tool generate_building_boundary executed.
9. Planner updated the task sequence: Generated a new geometry candidate.
10. Supervisor decision: report | Write the design report for the current best state.
11. Final report generated.

Agent final report:
## Bu

{'user_prompt': 'Generate an L-shaped building boundary with a building area of 3000 square meters. Rotate the building by 45 degrees. Use tool defaults for any unspecified shape parameters.',
 'workflow_mode': 'boundary_only',
 'decision_trace': ['Planner updated the task sequence: Initial planning required.',
  'Supervisor decision: read_site | Load site boundary, context, and legal constraints.',
  'Tool site_boundary_reader_04 executed.',
  'Tool legal_constraints_reader_04 executed.',
  'Tool context_reader_04 executed.',
  'Planner updated the task sequence: read_site step finished.',
  'Supervisor decision: generate_shape | Generating building 1 footprint boundary with the requested area (3000 sqm) and a single 45° rotation using tool defaults for other shape parameters.',
  'Tool generate_building_boundary executed.',
  'Planner updated the task sequence: Generated a new geometry candidate.',
  'Supervisor decision: report | Write the design report for the current best state.',

## Option B: live MCP site read and fit test

Use this section when the Swiftlet MCP server is already running and the Grasshopper tools `siteboundaryreader` and `testfit` are available.

The flow is:
1. Read the site boundary from Grasshopper through `siteboundaryreader`.
2. Reuse the Python-generated `gh_payload` building boundary.
3. Send both into `testfit` to check whether the building fits inside the live site boundary.

## Local end-to-end agent placement test

This section runs the actual Team 04 LangGraph agent from a user prompt. The agent generates the building boundary first, then runs constraint and evaluation steps, and only then executes the `place_building` node, which performs the centroid-guided fit loop before import.

The notebook now uses a reusable agent-side demo tool helper instead of defining notebook-only mock constraint and evaluation tools inline.

In [21]:
from __future__ import annotations

import importlib
import json
import os
import re
import socket
import urllib.error
import urllib.request

from langchain_openai import ChatOpenAI

from agent.notebook_demo_tools import build_notebook_demo_tool_client

USER_PLACEMENT_PROMPT = (
    "Generate an L-shaped building boundary with a building area of 3000 square meters. "
    "Then place it inside the site. Do not generate it inside the site directly; generate first and place after constraints pass."
    )

decision_engine_module = importlib.reload(importlib.import_module("agent.decision_engine"))
graph_module = importlib.reload(importlib.import_module("agent.graph"))
mcp_client_module = importlib.reload(importlib.import_module("agent.mcp_client"))
tool_catalog_module = importlib.reload(importlib.import_module("agent.tool_catalog"))
notebook_demo_tools_module = importlib.reload(importlib.import_module("agent.notebook_demo_tools"))

OpenAIDecisionEngine = decision_engine_module.OpenAIDecisionEngine
RuleBasedPlanner = decision_engine_module.RuleBasedPlanner
run_agent = graph_module.run_agent
CompositeToolClient = mcp_client_module.CompositeToolClient
LocalToolClient = mcp_client_module.LocalToolClient
build_default_local_tool_client = mcp_client_module.build_default_local_tool_client
ToolCatalog = tool_catalog_module.ToolCatalog
build_notebook_demo_tool_client = notebook_demo_tools_module.build_notebook_demo_tool_client
SITE_BOUNDARY_TOOL_NAME = globals().get("SITE_BOUNDARY_TOOL_NAME", "site_boundary_reader")
TEST_FIT_TOOL_NAME = globals().get("TEST_FIT_TOOL_NAME", "test_fit")

def build_live_mcp_tool_call_payload(tool_name: str, arguments: dict) -> dict:
    return {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": arguments,
        },
    }

def call_live_mcp_tool(tool_name: str, arguments: dict) -> dict:
    request_body = build_live_mcp_tool_call_payload(tool_name, arguments)
    request = urllib.request.Request(
        SWIFTLET_MCP_URL,
        data=json.dumps(request_body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read().decode("utf-8"))

def extract_mcp_content(response: dict) -> dict:
    result = response.get("result", response)
    if isinstance(result, dict):
        structured = result.get("structuredContent")
        if isinstance(structured, dict):
            return structured
        content = result.get("content")
        if isinstance(content, list):
            text_chunks = [
                item.get("text", "")
                for item in content
                if isinstance(item, dict) and item.get("type") == "text"
            ]
            if text_chunks:
                try:
                    return json.loads("\n".join(text_chunks))
                except json.JSONDecodeError:
                    return {"raw_text": "\n".join(text_chunks)}
    return result if isinstance(result, dict) else {"raw": result}

def extract_fit_flag(payload: dict) -> bool | None:
    if not isinstance(payload, dict):
        return None
    for key in ("IsFit", "is_fit", "fits", "fit", "fits_within_site_boundary"):
        value = payload.get(key)
        if isinstance(value, bool):
            return value
        if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
            return value.strip().lower() == "true"
    return None

agent_site_boundary = None
site_source = None
live_site_boundary_response = None
live_site_boundary_payload = None
live_site_boundary_error = None

try:
    live_site_boundary_response = call_live_mcp_tool(
        SITE_BOUNDARY_TOOL_NAME,
        {
            "site_guid": "TerraPilot_Input::SiteBoundary",
            "boundary": "",
        },
    )
    live_site_boundary_payload = extract_mcp_content(live_site_boundary_response)
    if isinstance(live_site_boundary_payload, dict) and isinstance(live_site_boundary_payload.get("site_boundary"), list):
        agent_site_boundary = live_site_boundary_payload["site_boundary"]
        site_source = "live_mcp_site_boundary_reader"
except urllib.error.URLError as exc:
    live_site_boundary_error = f"Swiftlet MCP request failed: {exc}"
except socket.timeout:
    live_site_boundary_error = "Swiftlet MCP request timed out after 60 seconds."

if agent_site_boundary is None and "SITE_BOUNDARY" in globals():
    agent_site_boundary = SITE_BOUNDARY
    site_source = "notebook_SITE_BOUNDARY_global"
elif agent_site_boundary is None and isinstance(globals().get("site_boundary_data"), dict) and isinstance(site_boundary_data.get("site_boundary"), list):
    agent_site_boundary = site_boundary_data["site_boundary"]
    site_source = "site_boundary_data"
elif agent_site_boundary is None and isinstance(globals().get("site_boundary_payload"), dict) and isinstance(site_boundary_payload.get("site_boundary"), list):
    agent_site_boundary = site_boundary_payload["site_boundary"]
    site_source = "site_boundary_payload"
elif agent_site_boundary is None:
    raise RuntimeError(
        "No usable site boundary coordinates were available from Rhino or the notebook kernel. "
        "Placement against test_fit requires a real site boundary."
    )

def live_test_fit(building_boundary: list[list[float]], site_boundary: list[list[float]]) -> dict:
    request_variants = [
        {
            "site_boundary": json.dumps(site_boundary),
            "building_boundary": json.dumps(building_boundary),
        },
        {
            "site_boundary": json.dumps({"site_boundary": site_boundary}),
            "building_boundary": json.dumps({"boundary": building_boundary}),
        },
    ]
    last_payload = None
    last_request_arguments = None
    last_error = None
    for request_arguments in request_variants:
        last_request_arguments = request_arguments
        try:
            response = call_live_mcp_tool(TEST_FIT_TOOL_NAME, request_arguments)
            payload = extract_mcp_content(response)
            last_payload = payload
            fit_flag = extract_fit_flag(payload if isinstance(payload, dict) else {})
            if fit_flag is not None:
                return {
                    "success": True,
                    "data": {
                        "IsFit": fit_flag,
                        "raw_payload": payload,
                        "request_arguments": request_arguments,
                    },
                    "metadata": {
                        "tool_name": TEST_FIT_TOOL_NAME,
                        "source": "live_mcp_proxy",
                    },
                }
        except urllib.error.URLError as exc:
            last_error = f"Swiftlet MCP request failed: {exc}"
        except socket.timeout:
            last_error = "Swiftlet MCP request timed out after 60 seconds."
    return {
        "success": True,
        "data": {
            "IsFit": False,
            "raw_payload": last_payload,
            "request_arguments": last_request_arguments,
            "error": last_error or "test_fit did not return a usable boolean result.",
        },
        "metadata": {
            "tool_name": TEST_FIT_TOOL_NAME,
            "source": "live_mcp_proxy",
        },
    }

live_validation_tool_client = LocalToolClient(
    {
        "test_fit": (
            {
                "name": "test_fit",
                "description": "Proxy the live Swiftlet MCP test_fit tool so placement uses the Rhino acceptance rule.",
            },
            live_test_fit,
        ),
    }
)

site_tool_client = build_notebook_demo_tool_client(agent_site_boundary)
tool_client = CompositeToolClient([build_default_local_tool_client(), site_tool_client, live_validation_tool_client])
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())
decision_engine = OpenAIDecisionEngine(
    llm=ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=0,
    )
)

placement_agent_state = run_agent(
    user_prompt=USER_PLACEMENT_PROMPT,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout={
        "site_boundary": agent_site_boundary,
        "target_building_count": 1,
        "workflow_mode": "full",
    },
    max_optimization_cycles=2,
    planner=RuleBasedPlanner(),
)

placed_buildings = placement_agent_state.get("placed_buildings", [])
if not placed_buildings:
    raise RuntimeError("The placement run did not return any placed building to import into Rhino/Grasshopper.")

placed_building = placed_buildings[-1]
live_import_arguments = {
    "geometry_id": placed_building["geometry_id"],
    "boundary": placed_building["boundary"],
    "layer_name": "TerraPilot_Output::BuildingFootprint",
    "closed": True,
}
live_import_request_body = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": GH_IMPORT_TOOL_NAME,
        "arguments": live_import_arguments,
    },
}

request = urllib.request.Request(
    SWIFTLET_MCP_URL,
    data=json.dumps(live_import_request_body).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

live_import_result = None
live_import_error = None
try:
    with urllib.request.urlopen(request, timeout=60) as response:
        live_import_result = json.loads(response.read().decode("utf-8"))
except urllib.error.URLError as exc:
    live_import_error = f"Swiftlet MCP request failed: {exc}"
except socket.timeout:
    live_import_error = "Swiftlet MCP request timed out after 60 seconds."

agent_step_trace = [
    message
    for message in placement_agent_state.get("messages", [])
    if message.startswith(("Planner updated", "Supervisor decision"))
]

placement_loop_trace = []
for record in placement_agent_state.get("tool_history", []):
    if not isinstance(record, dict):
        continue
    tool_name = record.get("tool")
    arguments = record.get("arguments", {}) if isinstance(record.get("arguments"), dict) else {}
    output = record.get("output", {}) if isinstance(record.get("output"), dict) else {}
    data = output.get("data", output) if isinstance(output, dict) else {}
    message = str(record.get("message", ""))

    if tool_name == "direction_to_site_centroid":
        match = re.search(r"iteration (\d+)", message)
        iteration_label = match.group(1) if match else "?"
        suggested_move = data.get("suggested_translate_by_xy", []) if isinstance(data, dict) else []
        distance_to_site = data.get("distance_to_site_centroid") if isinstance(data, dict) else None
        step_type = "move_toward_centroid"
        if "evaluated the placement candidate" in message:
            step_type = "post_transform_evaluation"
        placement_loop_trace.append(
            {
                "type": step_type,
                "iteration": iteration_label,
                "suggested_move": suggested_move,
                "distance_to_site_centroid": distance_to_site,
            }
        )
    elif tool_name == "modify_building_boundary":
        match = re.search(r"iteration (\d+)", message)
        iteration_label = match.group(1) if match else "?"
        placement_loop_trace.append(
            {
                "type": "try_transform",
                "iteration": iteration_label,
                "rotation_degrees": arguments.get("rotation_degrees"),
                "translate_by_xy": arguments.get("translate_by_xy"),
                "fits": data.get("fits_within_site_boundary") if isinstance(data, dict) else None,
            }
        )
    elif tool_name == "test_fit":
        placement_loop_trace.append(
            {
                "type": "live_test_fit",
                "fits": data.get("IsFit") if isinstance(data, dict) else None,
                "raw_payload": data.get("raw_payload") if isinstance(data, dict) else None,
                "error": data.get("error") if isinstance(data, dict) else None,
            }
        )

live_import_summary = {
    "tool_name": GH_IMPORT_TOOL_NAME,
    "geometry_id": live_import_arguments["geometry_id"],
    "point_count": len(live_import_arguments["boundary"]),
    "status": "ok" if live_import_result is not None else "error",
    "result": live_import_result,
    "error": live_import_error,
}

placement_summary = {
    "user_prompt": USER_PLACEMENT_PROMPT,
    "site_source": site_source,
    "site_boundary": agent_site_boundary,
    "live_site_boundary_payload": live_site_boundary_payload,
    "live_site_boundary_error": live_site_boundary_error,
    "agent_step_trace": agent_step_trace,
    "placement_loop_trace": placement_loop_trace,
    "placement_fit_summary": placement_agent_state.get("placement_fit_summary", {}),
    "placed_geometry_id": placed_building["geometry_id"],
    "live_import_summary": live_import_summary,
}

print(f"Site source used for placement: {site_source}")
if live_site_boundary_payload is not None:
    print("Live site boundary payload:")
    print(json.dumps(live_site_boundary_payload, indent=2))
elif live_site_boundary_error:
    print(f"Live site boundary read error: {live_site_boundary_error}")

print("\nAgent step trace:")
for index, step in enumerate(agent_step_trace, start=1):
    print(f"{index}. {step}")

print("\nPlacement loop trace:")
for index, step in enumerate(placement_loop_trace, start=1):
    if step["type"] == "move_toward_centroid":
        print(
            f"{index}. Loop {step['iteration']}: move toward site centroid by {step['suggested_move']} "
            f"| remaining distance {step['distance_to_site_centroid']}"
        )
    elif step["type"] == "post_transform_evaluation":
        print(
            f"{index}. Loop {step['iteration']}: evaluate transformed candidate "
            f"| remaining distance {step['distance_to_site_centroid']}"
        )
    elif step["type"] == "live_test_fit":
        print(
            f"{index}. Live test_fit validation: fits={step['fits']} "
            f"| payload={step['raw_payload']} | error={step['error']}"
        )
    else:
        print(
            f"{index}. Loop {step['iteration']}: try rotation {step['rotation_degrees']} deg "
            f"with translation {step['translate_by_xy']} | fits={step['fits']}"
        )

print("\nPlacement fit summary:")
print(json.dumps(placement_agent_state.get("placement_fit_summary", {}), indent=2))

print("\nLive import summary:")
print(json.dumps(live_import_summary, indent=2))

placement_summary

Site source used for placement: live_mcp_site_boundary_reader
Live site boundary payload:
{
  "site_boundary": [
    [
      161.21009405157622,
      324.78928297822404,
      0.0
    ],
    [
      78.60559594923592,
      144.61857334906495,
      0.0
    ],
    [
      341.31829298611035,
      133.855888671635,
      0.0
    ],
    [
      255.44277399979254,
      291.4651819915823,
      0.0
    ],
    [
      161.21009405157622,
      324.78928297822404,
      0.0
    ]
  ]
}

Agent step trace:
1. Planner updated the task sequence: Initial planning required.
2. Supervisor decision: read_site | Load site boundary, context, and legal constraints.
3. Planner updated the task sequence: read_site step finished.
4. Supervisor decision: generate_shape | Generate the next standalone geometry candidate for building 1 using the user-specified area (3000 sqm). Placement will occur in later steps after constraints pass.
5. Planner updated the task sequence: Generated a new geometry candida

{'user_prompt': 'Generate an L-shaped building boundary with a building area of 3000 square meters. Then place it inside the site. Do not generate it inside the site directly; generate first and place after constraints pass.',
 'site_source': 'live_mcp_site_boundary_reader',
 'site_boundary': [[161.21009405157622, 324.78928297822404, 0.0],
  [78.60559594923592, 144.61857334906495, 0.0],
  [341.31829298611035, 133.855888671635, 0.0],
  [255.44277399979254, 291.4651819915823, 0.0],
  [161.21009405157622, 324.78928297822404, 0.0]],
 'live_site_boundary_payload': {'site_boundary': [[161.21009405157622,
    324.78928297822404,
    0.0],
   [78.60559594923592, 144.61857334906495, 0.0],
   [341.31829298611035, 133.855888671635, 0.0],
   [255.44277399979254, 291.4651819915823, 0.0],
   [161.21009405157622, 324.78928297822404, 0.0]]},
 'live_site_boundary_error': None,
 'agent_step_trace': ['Planner updated the task sequence: Initial planning required.',
  'Supervisor decision: read_site | Load